# 提取关键帧图片

In [ ]:
from tqdm import tqdm
import os
import re
import cv2
from PIL import Image
import math
import numpy as np
import pickle

# 从图像中截取矩形区域
def get_rect_area(frame, rect):
    x, y, w, h = rect
    return frame[y:y+h, x:x+w]


def expand_rect_with_padding(rect, padding, limit_rect):
    """
    给矩形添加内边距扩大一圈，同时确保不超出给定limit_rect。

    :param rect: 原始矩形，格式为 (x, y, w, h)
    :param padding: 内边距，格式为 (pad_x, pad_y)
    :param limit_rect: 限制矩形，格式为 (limit_x, limit_y, limit_w, limit_h)
    :return: 新的矩形，格式为 (new_x, new_y, new_w, new_h)
    """
    x, y, w, h = rect
    pad_x, pad_y = padding
    limit_x, limit_y, limit_w, limit_h = limit_rect

    # 计算新矩形的位置和大小
    new_x = x - pad_x
    new_y = y - pad_y
    new_w = w + 2 * pad_x
    new_h = h + 2 * pad_y

    # 确保新矩形不超出limit_rect的左侧和顶部
    if new_x < limit_x:
        new_x = limit_x
        new_w = w + (x - limit_x) + pad_x
    if new_y < limit_y:
        new_y = limit_y
        new_h = h + (y - limit_y) + pad_y

    # 确保新矩形不超出limit_rect的右侧和底部
    if new_x + new_w > limit_x + limit_w:
        new_w = limit_x + limit_w - new_x
    if new_y + new_h > limit_y + limit_h:
        new_h = limit_y + limit_h - new_y

    return (new_x, new_y, new_w, new_h)


# 根据log文件找超声和造影区域，对应的label文件可能会替换
def get_area_from_log(analysis_log):
    rect_ultrasound = tuple()
    rect_contrast = tuple()

    # ()是分组，需要转义，\d+表示一个或多个数字
    with open(analysis_log, 'r') as f:
        for index, line in enumerate(f):
            if 10 < index < 18:
                pattern = r'\((\d+), (\d+), (\d+), (\d+)\)'
                matches = re.findall(pattern, line)
                if len(matches) == 2:
                    rects = [(int(x), int(y), int(w), int(h)) for x, y, w, h in matches]
                    rect_ultrasound = rects[0]
                    rect_contrast = rects[1]
                         
    return rect_ultrasound, rect_contrast


# 根据metrics.txt文件找关键帧时间戳
def get_metrics_from_log(metrics_path):
    roi1_TTP = 0
    roi2_TTP = 0
    target_TTP = 0
    width = 0
    height = 0
    frame_count = 0
    with open(metrics_path, 'r') as f:
        lines = f.readlines()
        fps = 0
        for line in lines:
            if 'fps' in line:
                fps = float(line.split(' ')[-1])
            if 'roi1' in line and 'TTP' in line:
                roi1_TTP = float(line.split(' ')[-1])
            if 'roi2' in line and 'TTP' in line:
                roi2_TTP = float(line.split(' ')[-1])
            if 'target' in line and 'TTP' in line:
                target_TTP = float(line.split(' ')[-1])
            if 'resolution' in line:
                width, height = map(int, line.split(' ')[1:])
            if 'frame_count' in line:
                frame_count = int(float(line.split(' ')[-1]))
                
        if math.isnan(roi1_TTP):
            roi1_TTP = roi2_TTP
        if math.isnan(roi2_TTP):
            roi2_TTP = roi1_TTP
            
        roi1_TTP = int(roi1_TTP * fps)
        roi2_TTP = int(roi2_TTP * fps)
        target_TTP = int(target_TTP * fps)
    
    return roi1_TTP, roi2_TTP, target_TTP, width, height, frame_count


def extract_frame(exp_path, output_path):
    os.makedirs(output_path, exist_ok=True)
    exp_name = os.path.basename(exp_path)

    # 关键帧超声区域保存路径
    ultrasound_roi1_path = os.path.join(output_path, 'ultrasound', 'roi1')
    ultrasound_target_path = os.path.join(output_path, 'ultrasound','target')
    os.makedirs(ultrasound_roi1_path, exist_ok=True)
    os.makedirs(ultrasound_target_path, exist_ok=True)

    # 关键帧造影区域保存路径
    contrast_roi1_path = os.path.join(output_path,'contrast', 'roi1')
    contrast_target_path = os.path.join(output_path, 'contrast', 'target')
    os.makedirs(contrast_roi1_path, exist_ok=True)
    os.makedirs(contrast_target_path, exist_ok=True)

    # 关键帧超声和造影合并保存路径
    all_roi1_path = os.path.join(output_path,'all', 'roi1')
    all_target_path = os.path.join(output_path, 'all', 'target')
    os.makedirs(all_roi1_path, exist_ok=True)
    os.makedirs(all_target_path, exist_ok=True)

    exps = os.listdir(exp_path)
    exps.sort()
    pbar = tqdm(exps)

    # 遍历实验中的每个视频
    for dir_name in pbar:

        if not os.path.isdir(os.path.join(exp_path, dir_name)):
            continue
        pbar.set_description(f"Processing {dir_name}")

        # 获取实验中的关键帧信息
        ttp_path = os.path.join(exp_path, dir_name, 'TTP')
        metrics_path = os.path.join(exp_path, dir_name, 'metrics.txt')
        analysis_log = os.path.join(exp_path, dir_name, 'analysis_log.txt')

        rect_ultrasound, rect_contrast = get_area_from_log(analysis_log)            
        roi1_TTP, roi2_TTP, target_TTP, width, height, frame_count = get_metrics_from_log(metrics_path)        
        history = pickle.load(open(os.path.join(exp_path, dir_name, 'history.pkl'), 'rb'))

        offset_x = rect_contrast[0] - rect_ultrasound[0]
        offset_y = rect_contrast[1] - rect_ultrasound[1]
        # print(dir_name, history[1].keys(), history[1]['rect'], offset_x, offset_y)

        padding = (100, 100)
        ###################################################################################################################
        # VueBox roi1_TTP contrast ultrasound
        frame = cv2.imread(os.path.join(ttp_path, 'roi1_TTP.jpg'))
        while history[roi1_TTP]['replaced_frame'] != 0:
            roi1_TTP = history[roi1_TTP]['replaced_frame']
        # cv2.polylines(frame, history[roi1_TTP]['polygon'], True, (0, 255, 0), 2)

        roi_ultrasound = history[roi1_TTP]['rect'][0]
        roi_contrast = [roi_ultrasound[0] + offset_x, roi_ultrasound[1] + offset_y, roi_ultrasound[2], roi_ultrasound[3]]
        
        roi1_peak_intensity = cv2.cvtColor(get_rect_area(frame, roi_contrast), cv2.COLOR_BGR2GRAY).mean()

        # 外接矩形加上一点边距
        roi_ultrasound = expand_rect_with_padding(roi_ultrasound, padding, rect_ultrasound)
        roi_contrast = expand_rect_with_padding(roi_contrast, padding, rect_contrast)

        cv2.imwrite(os.path.join(ultrasound_roi1_path, f'{dir_name}.jpg'), get_rect_area(frame, roi_ultrasound))
        cv2.imwrite(os.path.join(contrast_roi1_path, f'{dir_name}.jpg'), get_rect_area(frame, roi_contrast))

        # VueBox roi 超声造影叠加
        overlay_roi = cv2.addWeighted(
            get_rect_area(frame, roi_ultrasound), 0.5,
            get_rect_area(frame, roi_contrast), 0.5, 0,
        )
        cv2.imwrite(os.path.join(all_roi1_path, f"{dir_name}.jpg"), overlay_roi)

        ###################################################################################################################
        # Ours target_TTP contrast ultrasound
        frame = cv2.imread(os.path.join(ttp_path, 'target_TTP.jpg'))
        while history[target_TTP]['replaced_frame'] != 0:
            # print(target_TTP, history[target_TTP]['rect'])
            target_TTP = history[target_TTP]['replaced_frame']
        # cv2.polylines(frame, history[target_TTP]['polygon'], True, (0, 255, 0), 2)

        if target_TTP == 0 and history[target_TTP]['rect'] == []:
            print(target_TTP, history[target_TTP]['rect'])
            while history[target_TTP]['rect'] == []:
                target_TTP += 1 
                # target_TTP = history[target_TTP]['replaced_frame']
                print(target_TTP, history[target_TTP]['rect'])
            

        roi_ultrasound = history[target_TTP]['rect'][0]
        roi_contrast = [roi_ultrasound[0] + offset_x, roi_ultrasound[1] + offset_y, roi_ultrasound[2], roi_ultrasound[3]]
        
        target_peak_intensity = cv2.cvtColor(get_rect_area(frame, roi_contrast), cv2.COLOR_BGR2GRAY).mean()
        
        roi_ultrasound = expand_rect_with_padding(roi_ultrasound, padding, rect_ultrasound)
        roi_contrast = expand_rect_with_padding(roi_contrast, padding, rect_contrast)

        cv2.imwrite(os.path.join(ultrasound_target_path, f'{dir_name}.jpg'), get_rect_area(frame, roi_ultrasound))
        cv2.imwrite(os.path.join(contrast_target_path, f'{dir_name}.jpg'), get_rect_area(frame, roi_contrast))

        # target 超声造影叠加
        overlay_target = cv2.addWeighted(
            get_rect_area(frame, roi_ultrasound), 0.5,
            get_rect_area(frame, roi_contrast), 0.5, 0,
        )
        cv2.imwrite(os.path.join(all_target_path, f'{dir_name}.jpg'), overlay_target)

        ###################################################################################################################
        # 保存 peak_intensity
        with open(os.path.join(exp_path, dir_name, 'peak_intensity.txt'), 'w') as f:
            f.write(f'roi1_peak_intensity {roi1_peak_intensity:.4f}\n')
            f.write(f'target_peak_intensity {target_peak_intensity:.4f}\n')
        
        # 检查有没有保存不完整的
        if not os.path.exists(os.path.join(ultrasound_roi1_path, f'{dir_name}.jpg')):
            print(f'[{dir_name}] roi1_TTP key frame not found, roi1_TTP: {roi1_TTP}, roi2_TTP: {roi2_TTP}, target_TTP: {target_TTP}')

        if not os.path.exists(os.path.join(ultrasound_target_path, f'{dir_name}.jpg')):
            print(f'[{dir_name}] target_TTP key frame not found, roi1_TTP: {roi1_TTP}, roi2_TTP: {roi2_TTP}, target_TTP: {target_TTP}')


exp = 'qinzhou1'

exp_path = f"./exp/{exp}"
key_frame_path = f"./extract-key-frame/test/{exp}"
extract_frame(exp_path, key_frame_path)

# 划分标签

## 获取labels 良恶性 良性0 恶性1

In [ ]:
import os
import pandas as pd
import numpy as np

label_path = './data/labels'
path = os.listdir(label_path)
path = [os.path.join(label_path, i) for i in path]

dict = {}

for f in path:
    if 'hunan' in f:
        print(f)
        pf = pd.read_excel(f, sheet_name='Sheet1')
        hunan = pf.iloc[0:, [ord('A') - ord('A'), ord('K') - ord('A')]].values    
        for row in hunan:
            dict['image_'+ str(row[0])] = [str(row[1]), 'hunan']

    elif 'qinzhou1' in f:
        print(f)
        pf = pd.read_excel(f, sheet_name='Sheet1')
        qinzhou1 = pf.iloc[0:, [ord('A') - ord('A'), ord('B') - ord('A')]].values
        for row in qinzhou1:
            dict[str(row[0])] = [str(row[1]), 'qinzhou1']

    elif 'qinzhou2' in f:
        print(f)
        pf = pd.read_excel(f, sheet_name='Sheet1')
        qinzhou2 = pf.iloc[0:, [ord('A') - ord('A'), ord('I') - ord('A')]].values
        for row in qinzhou2:
            dict[str(row[0])] = [str(row[1]), 'qinzhou2']

    elif 'shihezi' in f:
        print(f)
        pf = pd.read_excel(f, sheet_name='Sheet1')
        shihezi = pf.iloc[0:, [ord('A') - ord('A'), ord('I') - ord('A')]].values     
        for row in shihezi:
            dict[str(row[0])] = [str(row[1]), 'shihezi']
dict["chenxinmei_"] = ["0", "qinzhou1"]
print(dict)
import json
dict = json.dumps(dict, indent=4, ensure_ascii=False)
with open('label.json', 'w') as f:
    f.write(dict)

## 划分良恶数据集

In [ ]:

import os, shutil

for path, dirs, files in os.walk(key_frame_path):
    if all(f in dirs for f in ('roi1', 'target')):
        print(path, dirs, files)
        
        # 先前创建过，不看这个文件夹了
        if os.path.exists(os.path.join(path, 'roi1', '0')):
            continue
        
        roi1 = os.path.join(path, 'roi1')
        # roi2 = os.path.join(path, 'roi2')
        target = os.path.join(path, 'target')
        
        os.makedirs(os.path.join(roi1, '0'), exist_ok=True)
        # os.makedirs(os.path.join(roi2, '0'), exist_ok=True)
        os.makedirs(os.path.join(target, '0'), exist_ok=True)
        
        os.makedirs(os.path.join(roi1, '1'), exist_ok=True)
        # os.makedirs(os.path.join(roi2, '1'), exist_ok=True)
        os.makedirs(os.path.join(target, '1'), exist_ok=True)
        
        l1 = os.listdir(roi1)
        # l2 = os.listdir(roi2)
        l3 = os.listdir(target)
        l1.sort()
        # l2.sort()
        l3.sort()
        assert l1 ==l3 , 'roi1, roi2, target should have the same files'
        
        for f1, f_target in zip(l1, l3):
            assert f1 == f_target, 'roi1, roi2, target should have the same name'
            name = os.path.splitext(f1)[0]
            if exp.__contains__('hunan'):
                name = "image_" + name
            if name in dict:
                shutil.move(os.path.join(roi1, f1), os.path.join(roi1, dict[name][0]))
                # shutil.move(os.path.join(roi2, f2), os.path.join(roi2, dict[name][0]))
                shutil.move(os.path.join(target, f_target), os.path.join(target, dict[name][0]))
            else:
                print(f'{f1} {name} not in dict')